# Prophet 
Kaggle Team Name: `[7] Material Girls`

Team Members:
- [564323] Eirill Bue
- [544590] Nora Langfeldt Borgenvik
- [586744] Silje Holm Johannesen

In [1]:
# Import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os, logging, warnings
from prophet import Prophet
import json
from datetime import timezone

# Base path for datasets
data_path = "data"

# Load data
receivals = pd.read_csv(f"{data_path}/kernel/receivals.csv")
purchase_orders = pd.read_csv(f"{data_path}/kernel/purchase_orders.csv")
materials = pd.read_csv(f"{data_path}/extended/materials.csv")
transportation = pd.read_csv(f"{data_path}/extended/transportation.csv")

# Convert date columns to datetime
receivals["date_arrival"] = pd.to_datetime(receivals["date_arrival"], utc=True)
purchase_orders["delivery_date"] = pd.to_datetime(purchase_orders["delivery_date"], utc=True)
purchase_orders["created_date_time"] = pd.to_datetime(purchase_orders["created_date_time"], utc=True)
purchase_orders["modified_date_time"] = pd.to_datetime(purchase_orders["modified_date_time"], utc=True)

In [2]:
# Remove duplicates
receivals = receivals.drop_duplicates()
purchase_orders = purchase_orders.drop_duplicates()

# Remove physically impossible values
receivals = receivals[receivals["net_weight"] > 0]
purchase_orders = purchase_orders[purchase_orders["quantity"] > 0]

# Standardize units to kilograms
if "unit" in purchase_orders.columns:
    purchase_orders["unit"] = purchase_orders["unit"].str.lower()
    purchase_orders.loc[purchase_orders["unit"].isin(["pund", "lbs", "pound"]), "quantity"] *= 0.45359237
    purchase_orders["unit"] = "kg"

# Ensure temporal consistency (receivals should not happen before order creation)
merged_temp = receivals.merge(
    purchase_orders[["purchase_order_id", "purchase_order_item_no", "created_date_time"]],
    on=["purchase_order_id", "purchase_order_item_no"],
    how="left"
)
receivals = merged_temp[merged_temp["date_arrival"] >= merged_temp["created_date_time"]].copy()
receivals.drop(columns="created_date_time", inplace=True)

# Keep only valid IDs
receivals = receivals[receivals["purchase_order_id"] > 0]
purchase_orders = purchase_orders[purchase_orders["purchase_order_id"] > 0]

# Merge datasets
training_data = pd.merge(
    receivals,
    purchase_orders,
    on=["purchase_order_id", "purchase_order_item_no"],
    how="inner"
)

# Keep only relevant columns for modeling
training_data = training_data[['rm_id', 'date_arrival', 'net_weight', 'quantity']]

# Floor date to day for aggregation
training_data['date_arrival'] = training_data['date_arrival'].dt.floor('D')

# Aggregate data by material and day
training_data = training_data.groupby(['rm_id', 'date_arrival'], as_index=False).agg(
    net_weight=('net_weight', 'sum'),
    quantity=('quantity', 'first')  # TODO: Check later
)
training_data = training_data.sort_values(['rm_id', 'date_arrival'])

# Save aggregated dataset as training_data.csv
training_data.to_csv(f"{data_path}/cleaned_data_prophet.csv", index=False)
print(f"✅ cleaning_data_prophet.csv created successfully with {len(training_data):,} rows.")

# Optional: Check for missing values
missing_summary = training_data.isnull().sum()
print(missing_summary)

✅ cleaning_data_prophet.csv created successfully with 40,519 rows.
rm_id           0
date_arrival    0
net_weight      0
quantity        0
dtype: int64


## The Model 

In [3]:
os.environ["CMDSTANPY_LOG"] = "CRITICAL"
for name in ["cmdstanpy","prophet","prophet.forecaster","prophet.models","prophet.diagnostics"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
    logging.getLogger(name).propagate = False
warnings.filterwarnings("ignore", message=".*cmdstanpy.*")

# ----------------------------
# Paths / Parameters
# ----------------------------
DATA_DIR = "data"
CLEAN_PATH = os.path.join(DATA_DIR, "cleaned_data_prophet.csv")
MAPPING_PATH = "data/prediction_mapping.csv"
SUBMISSION_OUT = "data/prophet_submission.csv"
RECEIVALS_RAW = os.path.join(DATA_DIR, "kernel", "receivals.csv")
PO_RAW        = os.path.join(DATA_DIR, "kernel", "purchase_orders.csv")

LAST_DELIVERY_CUTOFF = pd.Timestamp('2024-01-08')
PO_START  = pd.Timestamp('2024-10-01', tz='UTC')
PO_END    = pd.Timestamp('2025-12-31', tz='UTC')
FORECAST_START = pd.Timestamp('2025-01-01')
FORECAST_END   = pd.Timestamp('2025-05-31')

# ----------------------------
# 1) Load cleaned daily data
# ----------------------------
df = pd.read_csv(CLEAN_PATH)
df['date_arrival'] = pd.to_datetime(df['date_arrival'], utc=True).dt.tz_localize(None)
df = df.rename(columns={'date_arrival': 'date'})
df = df[['rm_id', 'date', 'net_weight']]

daily = (
    df.groupby(['rm_id', 'date'], as_index=False)['net_weight']
      .sum()
      .rename(columns={'net_weight': 'total_weight'})
)

# ----------------------------
# 2) Active rm_ids 
# ----------------------------
# A) based on last receival date
activity = daily.groupby('rm_id', as_index=False)['date'].agg(last_delivery='max')
active_by_receival = set(activity.loc[activity['last_delivery'] >= LAST_DELIVERY_CUTOFF, 'rm_id'].tolist())

# B) based on POs in window
active_by_pos, pos_error = None, None
try:
    if os.path.exists(RECEIVALS_RAW) and os.path.exists(PO_RAW):
        receivals_raw = pd.read_csv(RECEIVALS_RAW)
        po_raw = pd.read_csv(PO_RAW)

        receivals_raw['date_arrival'] = pd.to_datetime(receivals_raw['date_arrival'], utc=True)
        po_raw['delivery_date'] = pd.to_datetime(po_raw['delivery_date'], utc=True)

        req_cols_rec = {'purchase_order_id', 'purchase_order_item_no', 'date_arrival', 'rm_id'}
        req_cols_po  = {'purchase_order_id', 'purchase_order_item_no', 'delivery_date'}
        if not req_cols_rec.issubset(set(receivals_raw.columns)) or not req_cols_po.issubset(set(po_raw.columns)):
            raise KeyError("Missing required columns for PO-based active rm_ids.")

        last_receivals = (
            receivals_raw
            .sort_values('date_arrival')
            .groupby(['purchase_order_id', 'purchase_order_item_no'], as_index=False)
            .last()[['purchase_order_id', 'purchase_order_item_no', 'rm_id']]
        )

        po_merge = po_raw.merge(last_receivals, on=['purchase_order_id', 'purchase_order_item_no'], how='left')
        po_window = po_merge.loc[(po_merge['delivery_date'] >= PO_START) & (po_merge['delivery_date'] <= PO_END)]
        active_by_pos = set(po_window['rm_id'].dropna().unique().tolist())
    else:
        active_by_pos = None
except Exception as e:
    pos_error = str(e)
    print(f"[INFO] Skipping PO-based active rm_ids due to error: {e}")
    active_by_pos = None

A_list = sorted(list(active_by_receival))
print(f"\n[A] Active by receival >= {LAST_DELIVERY_CUTOFF.date()} (count={len(A_list)}):")
print(A_list)

if active_by_pos is None:
    msg = f"unavailable due to error: {pos_error}" if pos_error else "unavailable (files missing or not loaded)."
    print(f"\n[B] Active by POs {PO_START.date()} → {PO_END.date()}: {msg}")
    B_list = []
else:
    B_list = sorted(list(active_by_pos))
    print(f"\n[B] Active by POs {PO_START.date()} → {PO_END.date()} (count={len(B_list)}):")
    print(B_list)

# C) Stability filter on 2024
MIN_DAYS_2024     = 5
MIN_MONTHS_2024   = 3
MAX_MEAN_GAP_DAYS = 45
MAX_TOP_DAY_SHARE = 0.70

df_2024 = df.copy()
df_2024['date'] = pd.to_datetime(df_2024['date']).dt.tz_localize(None).dt.floor('D')
df_2024 = df_2024[(df_2024['date'] >= pd.Timestamp('2024-01-01')) &
                  (df_2024['date'] <= pd.Timestamp('2024-12-31'))]

days_24 = (df_2024[['rm_id','date']].drop_duplicates()
           .groupby('rm_id', as_index=False)['date'].count()
           .rename(columns={'date':'n_days_2024'}))

def mean_gap(g):
    d = np.sort(g['date'].unique())
    if len(d) <= 1:
        return np.inf
    gaps = np.diff(d) / np.timedelta64(1, 'D')
    return float(np.mean(gaps))

mean_gap_24 = (df_2024[['rm_id','date']].drop_duplicates()
               .groupby('rm_id', as_index=False)
               .apply(lambda x: pd.Series({'mean_gap_days': mean_gap(x)}))
               .reset_index(drop=True))

months_24 = (df_2024.assign(month=lambda x: x['date'].dt.month)
             .groupby(['rm_id','month'], as_index=False)['net_weight'].sum()
             .query('net_weight > 0')
             .groupby('rm_id', as_index=False)['month'].count()
             .rename(columns={'month':'months_active_2024'}))

daily_24  = df_2024.groupby(['rm_id','date'], as_index=False)['net_weight'].sum()
totals_24 = daily_24.groupby('rm_id', as_index=False)['net_weight'].sum().rename(columns={'net_weight':'total_2024'})
maxday_24 = daily_24.groupby('rm_id', as_index=False)['net_weight'].max().rename(columns={'net_weight':'max_day_2024'})
spike_24  = (totals_24.merge(maxday_24, on='rm_id', how='left')
             .assign(top_day_share=lambda x: np.where(x['total_2024']>0, x['max_day_2024']/x['total_2024'], 0.0))
             [['rm_id','top_day_share']])

metrics_24 = (days_24
              .merge(mean_gap_24, on='rm_id', how='outer')
              .merge(months_24,  on='rm_id', how='outer')
              .merge(spike_24,   on='rm_id', how='outer')
              .fillna({'n_days_2024':0, 'mean_gap_days':np.inf, 'months_active_2024':0, 'top_day_share':1.0}))

stable_mask = (
    (metrics_24['n_days_2024'] >= MIN_DAYS_2024) &
    (metrics_24['months_active_2024'] >= MIN_MONTHS_2024) &
    (metrics_24['mean_gap_days'] <= MAX_MEAN_GAP_DAYS) &
    (metrics_24['top_day_share'] <= MAX_TOP_DAY_SHARE)
)
C_set  = set(metrics_24.loc[stable_mask, 'rm_id'].tolist())
C_list = sorted(list(C_set))

print(f"\n[C] Stability filter (2024): days≥{MIN_DAYS_2024}, months≥{MIN_MONTHS_2024}, mean_gap≤{MAX_MEAN_GAP_DAYS}, top_day_share≤{MAX_TOP_DAY_SHARE}")
print(f"[C] Passing rm_ids (count={len(C_list)}):")
print(C_list[:50], '...' if len(C_list) > 50 else '')

# Final base set (A ∩ B) or A if B missing/empty
if (active_by_pos is not None) and (len(active_by_pos) > 0):
    base_set = set(active_by_receival).intersection(active_by_pos)
    if len(base_set) == 0:
        print("[INFO] PO ∩ Receival base empty; falling back to receival-based (A).")
        base_set = set(active_by_receival)
else:
    print("[INFO] Using receival-based active set only (A).")
    base_set = set(active_by_receival)

final_set = sorted(list(base_set.intersection(C_set)))
removed_by_C = sorted(list(base_set.difference(C_set)))
print(f"\n[Final] Active = Base ({len(base_set)}) ∩ Stability C ({len(C_set)}) → {len(final_set)} rm_ids")
print(f" - Removed by stability filter: {len(removed_by_C)}")
if removed_by_C:
    preview_cols = ['rm_id','n_days_2024','months_active_2024','mean_gap_days','top_day_share']
    print(metrics_24[metrics_24['rm_id'].isin(removed_by_C)]
          .sort_values('top_day_share', ascending=False)[preview_cols]
          .head(15).to_string(index=False))

# Apply filter
active_rm_ids = final_set
daily = daily[daily['rm_id'].isin(active_rm_ids)].copy()

# ----------------------------
# 3) Build complete panel
# ----------------------------
if daily.empty:
    raise RuntimeError("No active rm_ids found after filtering; adjust cutoff/window parameters.")

min_date    = daily['date'].min()
history_end = pd.Timestamp('2024-12-31')
date_idx    = pd.date_range(start=min_date, end=history_end, freq='D')

all_rm = np.sort(daily['rm_id'].unique())
grid = (pd.MultiIndex.from_product([all_rm, date_idx], names=['rm_id','date']).to_frame(index=False))
daily_full = grid.merge(daily, on=['rm_id','date'], how='left')
daily_full['total_weight'] = daily_full['total_weight'].fillna(0.0)

# ----------------------------
# 4) Deterministic regressors
# ----------------------------
def add_regressors(frame, date_col='date'):
    frame = frame.copy()
    ds = pd.to_datetime(frame[date_col])
    frame['is_weekend'] = (ds.dt.dayofweek >= 5).astype(int)
    m = ds.dt.month
    frame['month_sin'] = np.sin(2*np.pi*(m/12.0))
    frame['month_cos'] = np.cos(2*np.pi*(m/12.0))
    return frame

daily_full = add_regressors(daily_full, 'date')

# ----------------------------
# 5)
#     - Final model on full history → predict Jan–May 2025
#     - Backtest model train ≤ 2024-07-31 → predict Aug–Dec 2024
# ----------------------------
future_idx = pd.date_range(start=FORECAST_START, end=FORECAST_END, freq='D')
CUTOFF     = pd.Timestamp('2024-07-31')
TEST_START = pd.Timestamp('2024-08-01')
TEST_END   = pd.Timestamp('2024-12-31')
EPS        = 1e-6

pred_daily_rows = []
bt_rows = []

def _make_model():
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode='additive',
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=7.5
    )
    m.add_regressor('is_weekend', mode='multiplicative')
    m.add_regressor('month_sin',  mode='additive')
    m.add_regressor('month_cos',  mode='additive')
    return m

for rid in tqdm(all_rm, desc="Training & forecasting per rm_id", total=len(all_rm), dynamic_ncols=True):
    sub = (daily_full.loc[daily_full['rm_id']==rid, ['date','total_weight','is_weekend','month_sin','month_cos']]
           .sort_values('date').copy())

    # Final model → Jan–May 2025
    final_df = sub.rename(columns={'date':'ds','total_weight':'y'})
    try:
        m_final = _make_model()
        m_final.fit(final_df)

        fut = m_final.make_future_dataframe(periods=len(future_idx), freq='D', include_history=False).sort_values('ds')
        det = add_regressors(pd.DataFrame({'ds': future_idx}), 'ds')
        fut = fut.merge(det[['ds','is_weekend','month_sin','month_cos']], on='ds', how='left')

        fc = m_final.predict(fut)[['ds','yhat']].copy()
        fc['rm_id'] = rid
        fc['yhat']  = fc['yhat'].clip(lower=0.0)
        pred_daily_rows.append(fc.rename(columns={'ds':'date','yhat':'predicted_weight'}))
    except Exception:
        z = pd.DataFrame({'date': future_idx, 'predicted_weight': 0.0, 'rm_id': rid})
        pred_daily_rows.append(z)

    # Backtest model → Aug–Dec 2024 sum
    train_bt = sub[sub['date'] <= CUTOFF].rename(columns={'date':'ds','total_weight':'y'})
    true_bt  = sub[(sub['date']>=TEST_START) & (sub['date']<=TEST_END)]['total_weight'].sum()

    if (not train_bt.empty) and (train_bt['y'].sum() > 0):
        try:
            m_bt = _make_model()
            m_bt.fit(train_bt)

            fut_idx_bt = pd.date_range(TEST_START, TEST_END, freq='D')
            fut_bt = m_bt.make_future_dataframe(periods=len(fut_idx_bt), include_history=False, freq='D').sort_values('ds')
            det_bt = add_regressors(pd.DataFrame({'ds': fut_idx_bt}), 'ds')
            fut_bt = fut_bt.merge(det_bt[['ds','is_weekend','month_sin','month_cos']], on='ds', how='left')

            pred_bt_sum = m_bt.predict(fut_bt)['yhat'].clip(lower=0.0).sum()
        except Exception:
            pred_bt_sum = 0.0
    else:
        pred_bt_sum = 0.0

    bt_rows.append((rid, float(pred_bt_sum), float(true_bt)))

# Final daily preds
pred = pd.concat(pred_daily_rows, ignore_index=True)
pred['date'] = pd.to_datetime(pred['date'])

# ----------------------------
# 6) Conservative scaling (-15%)
# ----------------------------
pred['predicted_weight'] = pred['predicted_weight'].clip(lower=0) * 0.85
scaled_total = pred['predicted_weight'].sum()
print(f"📉 Total predicted weight (after 0.85 scaling): {scaled_total:,.2f}")

# ----------------------------
# 6c) Backtest down-only shrink (only for first-receival=2023)
# ----------------------------
bt = pd.DataFrame(bt_rows, columns=['rm_id','pred_augdec','true_augdec'])
bt['ratio_pred_to_true'] = (bt['pred_augdec'] / (bt['true_augdec'] + EPS)).replace([np.inf,-np.inf], np.nan)

BETA = 0.80
def dynamic_floor(r):
    xs = np.array([1.5, 3.0, 5.0]); ys = np.array([0.85, 0.60, 0.50])
    r = float(r)
    if r <= xs[0]: return 0.85
    if r >= xs[-1]: return 0.50
    return float(np.interp(r, xs, ys))

bt['raw_factor'] = (1.0 / (bt['ratio_pred_to_true'] + EPS)) ** BETA
bt['factor'] = 1.0
mask_over = bt['ratio_pred_to_true'] > 1.0
bt.loc[mask_over, 'factor'] = bt.loc[mask_over].apply(
    lambda row: max(dynamic_floor(row['ratio_pred_to_true']), min(1.0, row['raw_factor'])),
    axis=1
)

df_first = df.copy()
df_first['date'] = pd.to_datetime(df_first['date']).dt.tz_localize(None).dt.floor('D')
first_recv = (df_first.groupby('rm_id', as_index=False)['date'].min()
              .rename(columns={'date':'first_receival'}))
eligible_2023 = set(first_recv.loc[
    (first_recv['first_receival'] >= pd.Timestamp('2023-01-01')) &
    (first_recv['first_receival'] <= pd.Timestamp('2023-12-31')),
    'rm_id'
])
bt.loc[~bt['rm_id'].isin(eligible_2023), 'factor'] = 1.0

pred = pred.merge(bt[['rm_id','factor']], on='rm_id', how='left')
pred['factor'] = pred['factor'].fillna(1.0)
mask_fw = (pred['date'] >= pd.Timestamp('2025-01-01')) & (pred['date'] <= pd.Timestamp('2025-05-31'))
pred.loc[mask_fw, 'predicted_weight'] = pred.loc[mask_fw, 'predicted_weight'] * pred.loc[mask_fw, 'factor']
pred = pred.drop(columns=['factor'])

# Totals after downsize (overall + Jan–May window)
post_down_total = pred['predicted_weight'].sum()
post_down_janmay = pred.loc[
    (pred['date'] >= pd.Timestamp('2025-01-01')) & (pred['date'] <= pd.Timestamp('2025-05-31')),
    'predicted_weight'
].sum()
print(f"📦 Total predicted weight after downsize (Jan–May 2025): {post_down_janmay:,.2f}")

# ----------------------------
# 7) Aggregate to mapping windows → submission
# ----------------------------
mapping = pd.read_csv(MAPPING_PATH)
mapping['forecast_start_date'] = pd.to_datetime(mapping['forecast_start_date'])
mapping['forecast_end_date']   = pd.to_datetime(mapping['forecast_end_date'])

merged = mapping.merge(pred, on='rm_id', how='left')
mask = (merged['date'] >= merged['forecast_start_date']) & (merged['date'] <= merged['forecast_end_date'])
merged = merged.loc[mask]

submission = merged.groupby('ID', as_index=False)['predicted_weight'].sum()
all_ids = mapping[['ID']].drop_duplicates()
submission = all_ids.merge(submission, on='ID', how='left')
submission['predicted_weight'] = submission['predicted_weight'].fillna(0.0)
submission['predicted_weight'] = submission['predicted_weight'].round(0).astype(int)

submission_total = submission['predicted_weight'].sum()
print(f"🧾 Total predicted weight in submission (rounded): {submission_total:,.0f}")

submission.to_csv(SUBMISSION_OUT, index=False)
print(f"💾 Saved submission to: {SUBMISSION_OUT}")


[A] Active by receival >= 2024-01-08 (count=53):
[2123.0, 2124.0, 2125.0, 2129.0, 2130.0, 2131.0, 2132.0, 2133.0, 2134.0, 2135.0, 2140.0, 2142.0, 2143.0, 2144.0, 2145.0, 2147.0, 2161.0, 2741.0, 2981.0, 3121.0, 3122.0, 3123.0, 3124.0, 3125.0, 3126.0, 3142.0, 3201.0, 3265.0, 3282.0, 3362.0, 3381.0, 3421.0, 3461.0, 3581.0, 3601.0, 3621.0, 3642.0, 3701.0, 3761.0, 3781.0, 3865.0, 3883.0, 3901.0, 4021.0, 4081.0, 4222.0, 4263.0, 4302.0, 4401.0, 4441.0, 4443.0, 4481.0, 4501.0]

[B] Active by POs 2024-10-01 → 2025-12-31 (count=42):
[2123.0, 2125.0, 2129.0, 2130.0, 2131.0, 2132.0, 2133.0, 2134.0, 2135.0, 2142.0, 2143.0, 2144.0, 2145.0, 2161.0, 2741.0, 3122.0, 3123.0, 3124.0, 3125.0, 3126.0, 3282.0, 3362.0, 3381.0, 3421.0, 3581.0, 3601.0, 3701.0, 3781.0, 3865.0, 3883.0, 3901.0, 4081.0, 4161.0, 4222.0, 4263.0, 4302.0, 4441.0, 4461.0, 4462.0, 4463.0, 4481.0, 4501.0]

[C] Stability filter (2024): days≥5, months≥3, mean_gap≤45, top_day_share≤0.7
[C] Passing rm_ids (count=34):
[2129.0, 2130.0, 2131.0

C:\Users\silje\AppData\Local\Temp\ipykernel_29984\1674255306.py:113: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({'mean_gap_days': mean_gap(x)}))
Training & forecasting per rm_id:   0%|          | 0/27 [00:00<?, ?it/s]18:41:09 - cmdstanpy - INFO - Chain [1] start processing
18:41:10 - cmdstanpy - INFO - Chain [1] done processing
18:41:11 - cmdstanpy - INFO - Chain [1] start processing
18:41:11 - cmdstanpy - INFO - Chain [1] done processing
Training & forecasting per rm_id:   4%|▎         | 1/27 [00:03<01:37,  3.74s/it]18:41:12 - cmdstanpy - INFO - Chain [1] start processing
18:41:13 - cmdstanpy - INFO - Chain [1] done processing
18:41:13 - cmdstanpy - INFO - Chain [1]

📉 Total predicted weight (after 0.85 scaling): 29,026,761.21
📦 Total predicted weight after downsize (Jan–May 2025): 25,092,217.51
🧾 Total predicted weight in submission (rounded): 1,891,116,542
💾 Saved submission to: data/prophet_submission.csv


## Post-Processing

In [6]:
# ---- CONFIG ----
PRED_DF_NAME   = "pred"  
MAPPING_PATH   = "data/prediction_mapping.csv"
SUBMISSION_OUT = "data/prophet_submission.csv"
N_TARGET_DAYS  = 3       
FIFTHS_TOTAL   = 5       

# ---- Pull predictions ----
predictions = globals()[PRED_DF_NAME].copy()
predictions = predictions[['date','rm_id','predicted_weight']].copy()
predictions['date'] = pd.to_datetime(predictions['date']).dt.normalize()
predictions['predicted_weight'] = predictions['predicted_weight'].astype(float)

print(f"Baseline total (scaled daily): {predictions['predicted_weight'].sum():,.2f}")

# ---- Closed-day rule: weekends + Jan 1 + May 1 only ----
FIXED_CLOSED_DATES = {pd.Timestamp('2025-01-01'), pd.Timestamp('2025-05-01')}

def is_closed_day(ts: pd.Timestamp) -> bool:
    return (ts.weekday() >= 5) or (ts in FIXED_CLOSED_DATES)

# ---- Spread only 3/5 forward and remove 2/5 ----
def spread_three_fifths_and_remove_two(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values('date').copy()
    g['is_closed'] = g['date'].apply(is_closed_day)

    dates = g['date'].to_numpy()
    weights = g['predicted_weight'].to_numpy().astype(float)
    closed = g['is_closed'].to_numpy()
    is_open = ~closed

    removed_trace = np.zeros(len(g), dtype=float)

    for i in range(len(dates)):
        if closed[i] and weights[i] > 0:
            w = float(weights[i])
            share = w / FIFTHS_TOTAL  
            intended_distribute = N_TARGET_DAYS * share   
            intended_remove = 2 * share                   

            targets = []
            j = i + 1
            while j < len(dates) and len(targets) < N_TARGET_DAYS:
                if is_open[j]:
                    targets.append(j)
                j += 1

            actually_distributed = 0.0
            for idx in targets:
                weights[idx] += share
                actually_distributed += share

            weights[i] = 0.0

            undistributed_from_three_fifths = intended_distribute - actually_distributed
            actual_removed = intended_remove + max(0.0, undistributed_from_three_fifths)

            removed_trace[i] = actual_removed

    g['predicted_weight'] = weights
    g['removed_amount'] = removed_trace
    return g[['date','rm_id','predicted_weight','removed_amount']]

adjusted = (
    predictions
    .groupby('rm_id', group_keys=False)
    .apply(spread_three_fifths_and_remove_two)
    .reset_index(drop=True)
)

# ---- Totals & sanity ----
before_sum = predictions['predicted_weight'].sum()
after_sum  = adjusted['predicted_weight'].sum()
removed_sum = adjusted['removed_amount'].sum()
print("✅ Weekend + {Jan 1, May 1}: distributed 3/5, removed 2/5 (plus any undistributed)")
print(f"📊 Total weight before: {before_sum:,.2f}")
print(f"📉 Total weight after:  {after_sum:,.2f}")
print(f"🗑️  Total removed:       {removed_sum:,.2f}")
print(f"Conservation check (before - after ≈ removed): {abs((before_sum - after_sum) - removed_sum) < 1e-6}")

# ---- Rebuild submission using mapping ----
mapping = pd.read_csv(MAPPING_PATH)
mapping['forecast_start_date'] = pd.to_datetime(mapping['forecast_start_date'])
mapping['forecast_end_date']   = pd.to_datetime(mapping['forecast_end_date'])

merged = mapping.merge(adjusted[['date','rm_id','predicted_weight']], on='rm_id', how='left')
mask = (merged['date'] >= merged['forecast_start_date']) & (merged['date'] <= merged['forecast_end_date'])
merged = merged.loc[mask]

submission = merged.groupby('ID', as_index=False)['predicted_weight'].sum()

all_ids = mapping[['ID']].drop_duplicates()
submission = all_ids.merge(submission, on='ID', how='left')
submission['predicted_weight'] = submission['predicted_weight'].fillna(0.0)
submission['predicted_weight'] = submission['predicted_weight'].round(0).astype(int)

submission.to_csv(SUBMISSION_OUT, index=False)
print(f"💾 Saved adjusted submission (3/5 spread, 2/5 removed): {SUBMISSION_OUT}")

Baseline total (scaled daily): 25,092,217.51
✅ Weekend + {Jan 1, May 1}: distributed 3/5, removed 2/5 (plus any undistributed)
📊 Total weight before: 25,092,217.51
📉 Total weight after:  24,749,536.23
🗑️  Total removed:       342,681.28
Conservation check (before - after ≈ removed): True
💾 Saved adjusted submission (3/5 spread, 2/5 removed): data/prophet_submission.csv


C:\Users\silje\AppData\Local\Temp\ipykernel_29984\2444284504.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(spread_three_fifths_and_remove_two)
